# 01 - Bronze Demo: NYC Yellow Taxi

Notebook này minh họa PART B - Task 1 theo hướng Parquet-only.

- Đọc 12 monthly Parquet files và dirty Parquet fixture.
- Quan sát malformed dates, missing location IDs, missing coordinates và duplicates.
- Append các batch vào Delta Bronze.
- Kiểm tra metadata ingestion và `_delta_log/`.

Bronze không filter, deduplicate, update hoặc CDC. Các bước đó thuộc Silver.


In [9]:
from pathlib import Path
import sys

from pyspark.sql import functions as F


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "bronze" / "bronze_ingestion.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bronze.bronze_ingestion import create_spark, default_inputs, ingest_batch

SOURCE_DIR = PROJECT_ROOT / "data" / "source"
DIRTY_PATH = PROJECT_ROOT / "data" / "landing" / "dirty_test.parquet"
BRONZE_PATH = PROJECT_ROOT / "data" / "bronze" / "taxi_trips"
SOURCE_FILES = sorted(SOURCE_DIR.glob("yellow_tripdata_*.parquet"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Monthly source files: {len(SOURCE_FILES)}")
print(f"Dirty fixture exists: {DIRTY_PATH.exists()}")
print(f"Bronze path: {BRONZE_PATH}")


Project root: e:\Documents\DSEB.NEU.7TH\BIG DATA\New folder\delta-lakehouse-project
Monthly source files: 12
Dirty fixture exists: True
Bronze path: e:\Documents\DSEB.NEU.7TH\BIG DATA\New folder\delta-lakehouse-project\data\bronze\taxi_trips


In [10]:
def format_size(size_bytes: int) -> str:
    size = float(size_bytes)
    for unit in ("B", "KB", "MB", "GB"):
        if size < 1024 or unit == "GB":
            return f"{size:.2f} {unit}"
        size /= 1024


total_size = 0
for source_file in SOURCE_FILES:
    size_bytes = source_file.stat().st_size
    total_size += size_bytes
    print(f"- {source_file.name}: {format_size(size_bytes)}")
print(f"Total source size: {format_size(total_size)}")


- yellow_tripdata_2025-01.parquet: 56.42 MB
- yellow_tripdata_2025-02.parquet: 57.55 MB
- yellow_tripdata_2025-03.parquet: 66.72 MB
- yellow_tripdata_2025-04.parquet: 64.23 MB
- yellow_tripdata_2025-05.parquet: 74.23 MB
- yellow_tripdata_2025-06.parquet: 70.14 MB
- yellow_tripdata_2025-07.parquet: 63.84 MB
- yellow_tripdata_2025-08.parquet: 59.41 MB
- yellow_tripdata_2025-09.parquet: 69.08 MB
- yellow_tripdata_2025-10.parquet: 71.78 MB
- yellow_tripdata_2025-11.parquet: 67.84 MB
- yellow_tripdata_2025-12.parquet: 70.29 MB
Total source size: 791.52 MB


In [11]:
spark = create_spark()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")


Spark version: 3.5.9


## 1. Inspect the dirty Parquet fixture

Các lỗi được quan sát nhưng chưa được sửa trong Bronze.


In [12]:
dirty_df = spark.read.parquet(str(DIRTY_PATH)).cache()
pickup_ts = F.to_timestamp("tpep_pickup_datetime")
dropoff_ts = F.to_timestamp("tpep_dropoff_datetime")

quality = dirty_df.select(
    F.count("*").alias("rows"),
    F.sum(F.when(pickup_ts.isNull() & F.col("tpep_pickup_datetime").isNotNull(), 1).otherwise(0)).alias("malformed_pickup_dates"),
    F.sum(F.when(dropoff_ts.isNull() & F.col("tpep_dropoff_datetime").isNotNull(), 1).otherwise(0)).alias("malformed_dropoff_dates"),
    F.sum(F.when(F.col("PULocationID").isNull(), 1).otherwise(0)).alias("missing_pu_location"),
    F.sum(F.when(F.col("DOLocationID").isNull(), 1).otherwise(0)).alias("missing_do_location"),
    F.sum(F.when(
        F.col("pickup_latitude").isNull() | F.col("pickup_longitude").isNull() |
        F.col("dropoff_latitude").isNull() | F.col("dropoff_longitude").isNull(), 1
    ).otherwise(0)).alias("missing_coordinates"),
    F.sum(F.when(F.col("fare_amount") <= 0, 1).otherwise(0)).alias("invalid_fares"),
).first().asDict()

duplicate_groups = dirty_df.groupBy("trip_id").count().where(F.col("count") > 1)
print("Dirty fixture quality summary:")
for name, value in quality.items():
    print(f"- {name}: {value:,}")
print(f"- duplicate_trip_id_groups: {duplicate_groups.count():,}")
dirty_df.select(
    "trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "pickup_latitude", "dropoff_latitude"
).show(10, truncate=False)


Dirty fixture quality summary:
- rows: 5,218
- malformed_pickup_dates: 402
- malformed_dropoff_dates: 181
- missing_pu_location: 308
- missing_do_location: 276
- missing_coordinates: 1,154
- invalid_fares: 161
- duplicate_trip_id_groups: 309
+----------------------------------------------------------------+--------------------+---------------------+------------+------------+---------------+----------------+
|trip_id                                                         |tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|pickup_latitude|dropoff_latitude|
+----------------------------------------------------------------+--------------------+---------------------+------------+------------+---------------+----------------+
|40f3c9eae64201d2a2884a8bae212945f1ca77b79605b67c557f82feb253c0c7|not-a-date          |2025-99-99 25:61:00  |NULL        |NULL        |NULL           |NULL            |
|560e85c1bc15cc7050eba42715d0bfb9d20dd527c74b17964ddaa92e261c88b6|2025-01-01 00:32

## 2. Append all input batches to Bronze

Cell này bỏ qua batch ID đã có trong Delta table để notebook có thể chạy lại mà không append trùng ngoài ý muốn. Bản thân Bronze code vẫn dùng append-only.


In [13]:
BRONZE_PATH.parent.mkdir(parents=True, exist_ok=True)
existing_batch_ids = set()
if (BRONZE_PATH / "_delta_log").exists():
    existing_batch_ids = {
        row["ingest_batch_id"]
        for row in spark.read.format("delta").load(str(BRONZE_PATH))
        .select("ingest_batch_id").distinct().collect()
    }

for input_path in default_inputs():
    batch_id = input_path.stem
    if batch_id in existing_batch_ids:
        print(f"SKIP existing batch: {batch_id}")
        continue
    rows = ingest_batch(
        spark=spark,
        input_path=input_path,
        output_path=BRONZE_PATH,
        batch_id=batch_id,
    )
    print(f"APPENDED {rows:,} rows: {batch_id}")


SKIP existing batch: yellow_tripdata_2025-01
SKIP existing batch: yellow_tripdata_2025-02
SKIP existing batch: yellow_tripdata_2025-03
SKIP existing batch: yellow_tripdata_2025-04
SKIP existing batch: yellow_tripdata_2025-05
SKIP existing batch: yellow_tripdata_2025-06
SKIP existing batch: yellow_tripdata_2025-07
SKIP existing batch: yellow_tripdata_2025-08
SKIP existing batch: yellow_tripdata_2025-09
SKIP existing batch: yellow_tripdata_2025-10
SKIP existing batch: yellow_tripdata_2025-11
SKIP existing batch: yellow_tripdata_2025-12
SKIP existing batch: dirty_test


## 3. Inspect the Bronze Delta table


In [14]:
bronze_df = spark.read.format("delta").load(str(BRONZE_PATH)).cache()
print(f"Bronze rows: {bronze_df.count():,}")
bronze_df.groupBy("ingest_batch_id").count().orderBy("ingest_batch_id").show(20, truncate=False)
bronze_df.groupBy("record_source").count().show(truncate=False)
bronze_df.printSchema()
bronze_df.select(
    "trip_id", "tpep_pickup_datetime", "PULocationID", "DOLocationID",
    "record_source", "ingest_batch_id", "source_file", "raw_record_hash"
).show(10, truncate=False)


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "e:\Documents\DSEB.NEU.7TH\BIG DATA\spark_env\Lib\site-packages\pyspark\errors\exceptions\captured.py", line 179, in deco
    return f(*a, **kw)
  File "e:\Documents\DSEB.NEU.7TH\BIG DATA\spark_env\Lib\site-packages\py4j\protocol.py", line 327, in get_return_value
    raise Py4JJavaError(
        "An error occurred while calling {0}{1}{2}.\n".
        format(target_id, ".", name), value)
py4j.protocol.Py4JJavaError: <exception str() failed>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "e:\Documents\DSEB.NEU.7TH\BIG DATA\spark_env\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Ms Nhan\AppData\Local\Programs\Python\Python313\Lib\socket.py", line 719, in readinto
    return self._sock.recv_into(b)
  

Py4JError: org does not exist in the JVM

## 4. Verify that Bronze preserved dirty values

Silver sẽ xử lý các dòng này ở bước tiếp theo.


In [15]:
bronze_pickup_ts = F.to_timestamp("tpep_pickup_datetime")
bronze_df.where(
    (bronze_pickup_ts.isNull() & F.col("tpep_pickup_datetime").isNotNull())
    | F.col("PULocationID").isNull()
    | (F.col("fare_amount") <= 0)
).select(
    "trip_id", "tpep_pickup_datetime", "PULocationID", "DOLocationID", "fare_amount",
    "ingest_batch_id"
).show(10, truncate=False)


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

## 5. Inspect Delta transaction log

Mỗi batch append tạo một commit trong `_delta_log/`.


In [16]:
delta_log_files = sorted((BRONZE_PATH / "_delta_log").glob("*.json"))
print(f"Delta JSON commits: {len(delta_log_files)}")
for log_file in delta_log_files:
    print(log_file.name)


Delta JSON commits: 13
00000000000000000000.json
00000000000000000001.json
00000000000000000002.json
00000000000000000003.json
00000000000000000004.json
00000000000000000005.json
00000000000000000006.json
00000000000000000007.json
00000000000000000008.json
00000000000000000009.json
00000000000000000010.json
00000000000000000011.json
00000000000000000012.json


In [7]:
spark.stop()
